# Cross-Lingual Retrieval Corpus: Multi-Query, Difficulty-Aware Benchmark

**Part 1 (original pipeline)** builds a Nepali→English cross-lingual retrieval
corpus: Nepali Wikipedia titles are paired with their English Wikipedia
counterpart via Wikidata sitelinks, gold document extracts are fetched, and
uniformly-random distractor documents pad out the corpus.

**Part 2 (this extension)** upgrades that single-title-to-document benchmark
into a **multi-query, difficulty-aware** one:

1. **Multi-query formulations** — for every gold document, Wikidata aliases,
   Wikipedia redirects, and Wikidata short descriptions are pulled in
   alongside the original title, giving several semantically-equivalent
   Nepali query variants that all resolve to the *same* gold English
   document.
2. **Hard negatives** — instead of sampling distractors uniformly at random,
   we mine documents that share the gold entity's Wikidata class (via a
   SPARQL query), falling back to shared English Wikipedia categories when a
   class has too few members. These are topically close, so they are much
   harder to distinguish from the gold document than a random article.
3. **Per-instance metadata** — alias count, an ambiguity score (how many
   distinct Wikidata items share the query's label), the entity's Wikidata
   class, gold document length, and a lexical-overlap score between each
   query variant and its gold document, enabling slice-based analysis of
   retrieval performance (e.g. "does accuracy drop for highly ambiguous
   entities?" or "for entities with zero surface overlap?").

Everything in Part 2 runs on top of the checkpoints produced in Part 1
(`df_sample_clean`, `gold_docs`, `distractor_docs`, `corpus_df`) — re-run Part
1 first, or load its saved files, before continuing.

## Part 1 — Original corpus creation pipeline (unchanged)

### Step 1a. Quick test crawl: Nepali articles with an English langlink

In [1]:
import requests
import time
import json

API_URL = "https://ne.wikipedia.org/w/api.php"

HEADERS = {
    "User-Agent": "CLIR-research-script/1.0 (your_email@example.com; educational research project)"
}

def safe_get(session, params, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = session.get(API_URL, params=params, headers=HEADERS, timeout=30)
            if resp.status_code != 200:
                print(f"\nNon-200 status: {resp.status_code}, retrying... (attempt {attempt+1})")
                print("Response snippet:", resp.text[:300])
                time.sleep(2 ** attempt)
                continue
            return resp.json()
        except requests.exceptions.JSONDecodeError:
            print(f"\nJSON decode failed, retrying... (attempt {attempt+1})")
            print("Response snippet:", resp.text[:300])
            time.sleep(2 ** attempt)
        except requests.exceptions.RequestException as e:
            print(f"\nRequest exception: {e}, retrying... (attempt {attempt+1})")
            time.sleep(2 ** attempt)
    raise RuntimeError("Max retries exceeded — check response snippets above for the actual cause.")


def get_ne_en_pairs(limit=None, batch_size=500, sleep=0.3):
    pairs = []
    total_checked = 0
    params = {
        "action": "query",
        "format": "json",
        "generator": "allpages",
        "gapnamespace": 0,
        "gaplimit": batch_size,
        "prop": "langlinks",
        "lllang": "en",
        "lllimit": 1,
    }

    session = requests.Session()
    cont = {}

    while True:
        req_params = {**params, **cont}
        resp = safe_get(session, req_params)

        pages = resp.get("query", {}).get("pages", {})
        for pid, page in pages.items():
            total_checked += 1
            ne_title = page.get("title")
            langlinks = page.get("langlinks")
            if langlinks:
                en_title = langlinks[0]["*"]
                pairs.append({
                    "pageid": pid,
                    "ne_title": ne_title,
                    "en_title": en_title
                })

        print(f"Checked {total_checked} articles, found {len(pairs)} en-linked so far", end="\r")

        if limit and total_checked >= limit:
            break

        if "continue" in resp:
            cont = resp["continue"]
        else:
            break

        time.sleep(sleep)

    return pairs

# Quick test first with a small limit before running the full crawl
pairs = get_ne_en_pairs(limit=1000, batch_size=500, sleep=0.3)
print(f"\n\nTest run — checked ~1000, found {len(pairs)} pairs")
for p in pairs[:5]:
    print(p)

Checked 1000 articles, found 2 en-linked so far

Test run — checked ~1000, found 2 pairs
{'pageid': '839', 'ne_title': 'अधिवक्ता', 'en_title': 'Lawyer'}
{'pageid': '1908', 'ne_title': 'अजरबैजान', 'en_title': 'Azerbaijan'}


### Step 1b. Full crawl helpers: Nepali pages → QIDs → English sitelinks

In [2]:
import requests
import time
import json

NE_API = "https://ne.wikipedia.org/w/api.php"
WD_API = "https://www.wikidata.org/w/api.php"

HEADERS = {
    "User-Agent": "CLIR-research-script/1.0 (your_email@example.com; educational research project)"
}

def safe_get(url, params, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
            if resp.status_code != 200:
                print(f"\nNon-200: {resp.status_code}, retry {attempt+1}")
                time.sleep(2 ** attempt)
                continue
            return resp.json()
        except requests.exceptions.JSONDecodeError:
            print(f"\nJSON decode failed, retry {attempt+1}. Snippet: {resp.text[:200]}")
            time.sleep(2 ** attempt)
        except requests.exceptions.RequestException as e:
            print(f"\nRequest exception: {e}, retry {attempt+1}")
            time.sleep(2 ** attempt)
    raise RuntimeError("Max retries exceeded")


def get_ne_pages_with_qids(limit=None, batch_size=500, sleep=0.2):
    """Step 1: crawl all Nepali articles and get their Wikidata QID."""
    pages_out = []
    total_checked = 0
    params = {
        "action": "query",
        "format": "json",
        "generator": "allpages",
        "gapnamespace": 0,
        "gaplimit": batch_size,
        "prop": "pageprops",
        "ppprop": "wikibase_item",
    }
    cont = {}
    while True:
        req_params = {**params, **cont}
        resp = safe_get(NE_API, req_params)
        pages = resp.get("query", {}).get("pages", {})
        for pid, page in pages.items():
            total_checked += 1
            qid = page.get("pageprops", {}).get("wikibase_item")
            if qid:
                pages_out.append({
                    "pageid": pid,
                    "ne_title": page.get("title"),
                    "qid": qid
                })
        print(f"Checked {total_checked} ne articles, {len(pages_out)} have QIDs", end="\r")

        if limit and total_checked >= limit:
            break
        if "continue" in resp:
            cont = resp["continue"]
        else:
            break
        time.sleep(sleep)
    return pages_out


def batch_check_enwiki_sitelinks(qid_records, batch_size=50, sleep=0.2):
    """Step 2: batch-query Wikidata for enwiki sitelinks."""
    pairs = []
    qids = [r["qid"] for r in qid_records]
    qid_to_record = {r["qid"]: r for r in qid_records}

    for i in range(0, len(qids), batch_size):
        batch = qids[i:i+batch_size]
        params = {
            "action": "wbgetentities",
            "format": "json",
            "ids": "|".join(batch),
            "props": "sitelinks",
            "sitefilter": "enwiki",
        }
        resp = safe_get(WD_API, params)
        entities = resp.get("entities", {})
        for qid, ent in entities.items():
            sitelinks = ent.get("sitelinks", {})
            if "enwiki" in sitelinks:
                rec = qid_to_record[qid]
                pairs.append({
                    "pageid": rec["pageid"],
                    "ne_title": rec["ne_title"],
                    "qid": qid,
                    "en_title": sitelinks["enwiki"]["title"]
                })
        print(f"Processed {min(i+batch_size, len(qids))}/{len(qids)} QIDs, {len(pairs)} en pairs found", end="\r")
        time.sleep(sleep)

    return pairs


# --- Test run first ---
test_pages = get_ne_pages_with_qids(limit=1000, batch_size=500, sleep=0.2)
print(f"\n\nGot {len(test_pages)} pages with QIDs out of 1000 checked")

test_pairs = batch_check_enwiki_sitelinks(test_pages, batch_size=50, sleep=0.2)
print(f"\n\nOf those, {len(test_pairs)} have an English Wikipedia sitelink")
for p in test_pairs[:10]:
    print(p)

Checked 1000 ne articles, 719 have QIDs

Got 719 pages with QIDs out of 1000 checked
Processed 719/719 QIDs, 659 en pairs found

Of those, 659 have an English Wikipedia sitelink
{'pageid': '102605', 'ne_title': '...रेडि फर इट?', 'qid': 'Q38552445', 'en_title': '...Ready for It?'}
{'pageid': '4323', 'ne_title': 'अ', 'qid': 'Q22947279', 'en_title': 'A (Indic)'}
{'pageid': '107378', 'ne_title': 'अ क्लकवर्क अरेञ्ज (चलचित्र)', 'qid': 'Q181086', 'en_title': 'A Clockwork Orange (film)'}
{'pageid': '117606', 'ne_title': 'अ जर्नी अफ सम्यक बुद्ध', 'qid': 'Q15059369', 'en_title': 'A Journey of Samyak Buddha'}
{'pageid': '51546', 'ne_title': 'अ पकेट फुल अफ राइ', 'qid': 'Q29132', 'en_title': 'A Pocket Full of Rye'}
{'pageid': '45994', 'ne_title': 'अ ब्रिफ हिस्ट्री अफ टाइम', 'qid': 'Q471726', 'en_title': 'A Brief History of Time'}
{'pageid': '51937', 'ne_title': 'अ स्क्यानर डार्क्ली', 'qid': 'Q1198489', 'en_title': 'A Scanner Darkly'}
{'pageid': '44315', 'ne_title': 'अ स्ट्रेन्जर इन द मिरर्', 'qid':

### Step 1c. Full resumable crawl + Wikidata batch lookup

In [3]:
import os

CHECKPOINT_PAGES = "ne_pages_with_qids.json"
CHECKPOINT_PAIRS = "ne_en_pairs.json"

# --- Step 1: full crawl of Nepali articles + QIDs (resumable is optional here since
# this step is fast; the expensive/fragile part is Step 2's many Wikidata batch calls) ---
all_pages = get_ne_pages_with_qids(limit=None, batch_size=500, sleep=0.2)
print(f"\n\nTotal ne articles checked, {len(all_pages)} have QIDs")

with open(CHECKPOINT_PAGES, "w", encoding="utf-8") as f:
    json.dump(all_pages, f, ensure_ascii=False, indent=2)
print(f"Saved checkpoint: {CHECKPOINT_PAGES}")


# --- Step 2: batch Wikidata lookup, with incremental saving every N batches ---
def batch_check_enwiki_sitelinks_resumable(qid_records, batch_size=50, sleep=0.2,
                                             save_every=20, save_path=CHECKPOINT_PAIRS):
    pairs = []
    qids = [r["qid"] for r in qid_records]
    qid_to_record = {r["qid"]: r for r in qid_records}
    n_batches = (len(qids) + batch_size - 1) // batch_size

    for batch_idx, i in enumerate(range(0, len(qids), batch_size)):
        batch = qids[i:i+batch_size]
        params = {
            "action": "wbgetentities",
            "format": "json",
            "ids": "|".join(batch),
            "props": "sitelinks",
            "sitefilter": "enwiki",
        }
        resp = safe_get(WD_API, params)
        entities = resp.get("entities", {})
        for qid, ent in entities.items():
            sitelinks = ent.get("sitelinks", {})
            if "enwiki" in sitelinks:
                rec = qid_to_record[qid]
                pairs.append({
                    "pageid": rec["pageid"],
                    "ne_title": rec["ne_title"],
                    "qid": qid,
                    "en_title": sitelinks["enwiki"]["title"]
                })

        print(f"Batch {batch_idx+1}/{n_batches}, {len(pairs)} pairs so far", end="\r")

        if (batch_idx + 1) % save_every == 0:
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(pairs, f, ensure_ascii=False, indent=2)

        time.sleep(sleep)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(pairs, f, ensure_ascii=False, indent=2)
    return pairs


final_pairs = batch_check_enwiki_sitelinks_resumable(all_pages, batch_size=50, sleep=0.2)
print(f"\n\nFinal total ne<->en pairs: {len(final_pairs)}")

Checked 42434 ne articles, 30689 have QIDs

Total ne articles checked, 30689 have QIDs
Saved checkpoint: ne_pages_with_qids.json
Batch 614/614, 26431 pairs so far

Final total ne<->en pairs: 26431


### Step 1d. Clean and deduplicate raw pairs

In [4]:
import pandas as pd

df = pd.DataFrame(final_pairs)
print("Raw pairs:", len(df))

# Drop exact duplicate en_titles (rare, but possible if multiple ne pages map oddly)
df = df.drop_duplicates(subset="en_title")
print("After en_title dedup:", len(df))

# Drop duplicate ne_titles just in case
df = df.drop_duplicates(subset="ne_title")
print("After ne_title dedup:", len(df))

df.to_csv("ne_en_pairs_clean.csv", index=False)
df.head(10)

Raw pairs: 26431
After en_title dedup: 26431
After ne_title dedup: 26431


,pageid,ne_title,qid,en_title
0,102605,...रेडि फर इट?,Q38552445,...Ready for It?
1,4323,अ,Q22947279,A (Indic)
2,107378,अ क्लकवर्क अरेञ्ज (चलचित्र),Q181086,A Clockwork Orange (film)
3,117606,अ जर्नी अफ सम्यक बुद्ध,Q15059369,A Journey of Samyak Buddha
4,51546,अ पकेट फुल अफ राइ,Q29132,A Pocket Full of Rye
5,45994,अ ब्रिफ हिस्ट्री अफ टाइम,Q471726,A Brief History of Time
6,51937,अ स्क्यानर डार्क्ली,Q1198489,A Scanner Darkly
7,44315,अ स्ट्रेन्जर इन द मिरर्,Q2690102,A Stranger in the Mirror
8,126439,अँगालो,Q328703,Hug
9,53292,अँडिर,Q155867,Ricinus


### Step 1e. Filter to native-script Nepali titles and sample the query pool

In [5]:
import pandas as pd
import re
import random

if 'final_pairs' in dir():
    df = pd.DataFrame(final_pairs)
else:
    with open("ne_en_pairs.json", "r", encoding="utf-8") as f:
        df = pd.DataFrame(json.load(f))

print("Raw pairs:", len(df))
df = df.drop_duplicates(subset="en_title").drop_duplicates(subset="ne_title").reset_index(drop=True)
print("After dedup:", len(df))

def is_mostly_latin(text, threshold=0.5):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return False
    latin = sum(1 for c in letters if ord(c) < 0x0250)
    return (latin / len(letters)) >= threshold

df["ne_title_is_latinized"] = df["ne_title"].apply(is_mostly_latin)
print(df["ne_title_is_latinized"].value_counts())

df_native = df[~df["ne_title_is_latinized"]].reset_index(drop=True)
print(f"\nNative Nepali-script queries: {len(df_native)}")

random.seed(42)

TARGET_N = 10_000
POOL_N = 12_000

assert POOL_N >= TARGET_N
df_pool = df_native.sample(n=min(POOL_N, len(df_native)), random_state=42).reset_index(drop=True)
print(f"Candidate pool size: {len(df_pool)}  (target: {TARGET_N})")

df_pool.to_csv("query_pairs_pool.csv", index=False)
df_pool.head(10)

Raw pairs: 26431
After dedup: 26431
ne_title_is_latinized
False    26431
Name: count, dtype: int64

Native Nepali-script queries: 26431
Candidate pool size: 12000  (target: 10000)


,pageid,ne_title,qid,en_title,ne_title_is_latinized
0,7137,तेश्रो नेपाल तिब्बत युद्ध,Q2001528,Nepal–Tibet War (1855–1856),False
1,137575,हर हर महादेव,Q119038744,Hara Hara Mahadeva,False
2,104178,कुमाइँ,Q3630074,Kumaoni people,False
3,80513,म्याग्गी जिलेनहाल,Q202381,Maggie Gyllenhaal,False
4,78074,हाहोए लोक गाँउ,Q45865,Hahoe Folk Village,False
5,81823,न्युजिल्यान्ड विरुद्ध श्रीलङ्का २०१५-१६,Q20648934,Sri Lankan cricket team in New Zealand in 2015–16,False
6,78160,शिरवंशाहको दरबार,Q338889,Palace of the Shirvanshahs,False
7,48034,सरस्वती नदी,Q177321,Saraswati River,False
8,78589,संयुक्त राज्य अमेरिकाका विश्व सम्पदा क्षेत्रहर...,Q2623809,List of World Heritage Sites in the United States,False
9,56850,ताल्चा,Q228039,Lock and key,False


### Step 1f. English extract-fetching helper (test run)

In [6]:
import requests
import time
import json

EN_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "CLIR-research-script/1.0 (your_email@example.com; educational research project)"
}

def safe_get(url, params, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
            if resp.status_code != 200:
                print(f"\nNon-200: {resp.status_code}, retry {attempt+1}")
                time.sleep(2 ** attempt)
                continue
            data = resp.json()
            if "error" in data:
                # Surface the actual API error instead of silently returning nothing
                print(f"\nAPI error: {data['error']}")
                raise RuntimeError(f"MediaWiki API error: {data['error']}")
            return data
        except requests.exceptions.JSONDecodeError:
            print(f"\nJSON decode failed, retry {attempt+1}. Snippet: {resp.text[:200]}")
            time.sleep(2 ** attempt)
        except requests.exceptions.RequestException as e:
            print(f"\nRequest exception: {e}, retry {attempt+1}")
            time.sleep(2 ** attempt)
    raise RuntimeError("Max retries exceeded")

def truncate_tokens(text, n_tokens=200):
    tokens = text.split()
    return " ".join(tokens[:n_tokens])

def fetch_extracts(titles, batch_size=50):
    """Fetch extracts in batches, since the API caps 'titles' at 50 per request
    for non-bot accounts (500 for bots)."""
    extracts = {}
    for i in range(0, len(titles), batch_size):
        batch = titles[i:i + batch_size]
        params = {
            "action": "query",
            "format": "json",
            "titles": "|".join(batch),
            "prop": "extracts",
            "explaintext": 1,
            "exintro": 0,
            "redirects": 1,
        }
        resp = safe_get(EN_API, params)
        pages = resp.get("query", {}).get("pages", {})
        batch_extracts = {p.get("title"): p.get("extract", "") for p in pages.values() if p.get("extract")}
        extracts.update(batch_extracts)
        print(f"Batch {i//batch_size + 1}: {len(batch_extracts)}/{len(batch)} fetched")
    return extracts

test_titles = df_pool["en_title"].tolist()[:100]
test_extracts = fetch_extracts(test_titles)
print(f"\nTest: {len(test_extracts)}/100 fetched successfully")

for title in list(test_extracts.keys())[:3]:
    text = test_extracts[title]
    print(f"\n--- {title} ({len(text.split())} tokens) ---")
    print(text[:300])

Batch 1: 20/50 fetched
Batch 2: 20/50 fetched

Test: 40/100 fetched successfully

--- 2018 ICC World Cricket League Division Two (230 tokens) ---
2018 ICC World Cricket League Division Two was a cricket tournament that took place in February 2018 in Namibia. The United Arab Emirates won the tournament, after beating Nepal by 7 runs in the final. Canada and Namibia finished third and fourth respectively and remained in Division Two. Oman and K

--- Amchaur (39 tokens) ---
Amchaur is a small town and Village Development Committee in Baitadi District in Sudurpashchim Province of western Nepal. At the time of the 1991 Nepal census it had a population of 3,852 and had 700 houses in the town.

--- Amit Shah (429 tokens) ---
Amit Anilchandra Shah (born 22 October 1964) is an Indian politician who is serving as the 31st Minister of Home Affairs since 2019, the longest-serving holder of the office in Indian history. Additionally he has been the 1st Minister of Co-operation since July 2021. He i

### Step 1g. Collect gold documents up to `TARGET_N`

In [7]:
gold_docs = {}
batch_size = 20
sleep = 0.2

pool_titles = df_pool["en_title"].tolist()
n_pool = len(pool_titles)
i = 0

while len(gold_docs) < TARGET_N and i < n_pool:
    batch = pool_titles[i:i + batch_size]
    i += batch_size

    params = {
        "action": "query",
        "format": "json",
        "titles": "|".join(batch),
        "prop": "extracts",
        "explaintext": 1,
        "exintro": 0,
        "redirects": 1,
    }
    resp = safe_get(EN_API, params)
    query_result = resp.get("query", {})
    pages = query_result.get("pages", {})

    redirect_map = {r["from"]: r["to"] for r in query_result.get("redirects", [])}
    extract_by_resolved_title = {
        page.get("title"): page.get("extract", "")
        for page in pages.values()
        if page.get("title") and page.get("extract")
    }

    for orig_title in batch:
        if len(gold_docs) >= TARGET_N:
            break
        resolved_title = redirect_map.get(orig_title, orig_title)
        extract = extract_by_resolved_title.get(resolved_title)
        if extract and orig_title not in gold_docs:
            gold_docs[orig_title] = truncate_tokens(extract, 200)

    print(f"Checked {min(i, n_pool)}/{n_pool} pool titles, {len(gold_docs)}/{TARGET_N} gold docs collected", end="\r")
    time.sleep(sleep)

print(f"\n\nFinal gold doc count: {len(gold_docs)} (target {TARGET_N})")

if len(gold_docs) < TARGET_N:
    raise RuntimeError(
        f"Pool exhausted before reaching target ({len(gold_docs)}/{TARGET_N}). "
        f"Increase POOL_N in the sampling cell above and re-run."
    )

lengths = [len(t.split()) for t in gold_docs.values()]
print(f"Min/Mean/Max tokens: {min(lengths)} / {sum(lengths)/len(lengths):.1f} / {max(lengths)}")

with open("gold_docs_truncated.json", "w", encoding="utf-8") as f:
    json.dump(gold_docs, f, ensure_ascii=False, indent=2)

df_sample_clean = df_pool[df_pool["en_title"].isin(gold_docs.keys())].reset_index(drop=True)
df_sample_clean = df_sample_clean.drop_duplicates(subset="en_title").reset_index(drop=True)
df_sample_clean.to_csv("query_pairs_final.csv", index=False)
print(f"Final query set: {len(df_sample_clean)} rows")

Checked 10020/12000 pool titles, 10000/10000 gold docs collected

Final gold doc count: 10000 (target 10000)
Min/Mean/Max tokens: 4 / 114.9 / 200
Final query set: 10000 rows


### Step 1h. Baseline distractors — uniformly random English articles

In [8]:
import random
import time
import json

EN_API = "https://en.wikipedia.org/w/api.php"

print("Starting collection..")

def fetch_random_en_articles(n_needed, batch_size=20, sleep=0.2,
                               exclude_titles=None, save_every=100,
                               save_path="distractor_docs.json"):
    exclude_titles = exclude_titles or set()
    results = {}
    seen_titles = set()

    while len(results) < n_needed:
        rand_params = {
            "action": "query",
            "format": "json",
            "list": "random",
            "rnnamespace": 0,
            "rnlimit": batch_size,
        }
        rand_resp = safe_get(EN_API, rand_params)
        random_pages = rand_resp.get("query", {}).get("random", [])
        titles = [p["title"] for p in random_pages
                  if p["title"] not in exclude_titles and p["title"] not in seen_titles]

        if not titles:
            continue

        for t in titles:
            seen_titles.add(t)

        extract_params = {
            "action": "query",
            "format": "json",
            "titles": "|".join(titles),
            "prop": "extracts",
            "explaintext": 1,
            "exintro": 0,
            "redirects": 1,
        }
        resp = safe_get(EN_API, extract_params)
        pages = resp.get("query", {}).get("pages", {})
        for pid, page in pages.items():
            title = page.get("title")
            extract = page.get("extract", "")
            if title and extract and len(extract.split()) >= 20 and title not in exclude_titles:
                results[title] = extract

        print(f"Collected {len(results)}/{n_needed} distractor docs", end="\r")

        if len(results) % save_every < batch_size:
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

        time.sleep(sleep)

    results = dict(list(results.items())[:n_needed])
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return results


N_DISTRACTORS = 20_000
gold_titles_set = set(gold_docs.keys())

distractor_docs_raw = fetch_random_en_articles(
    n_needed=N_DISTRACTORS, batch_size=20, sleep=0.15,
    exclude_titles=gold_titles_set, save_every=500
)

assert len(distractor_docs_raw) == N_DISTRACTORS
print(f"\n\nFinal distractor count: {len(distractor_docs_raw)}")

distractor_docs = {title: truncate_tokens(text, 200) for title, text in distractor_docs_raw.items()}
lengths = [len(t.split()) for t in distractor_docs.values()]
print(f"Distractor token stats — Min/Mean/Max: {min(lengths)} / {sum(lengths)/len(lengths):.1f} / {max(lengths)}")

Starting collection..
Collected 20011/20000 distractor docs

Final distractor count: 20000
Distractor token stats — Min/Mean/Max: 20 / 77.0 / 200


### Step 1i. Assemble & save the baseline (single-query, random-distractor) corpus

This reproduces the original benchmark as-is. Part 2 below builds a second,
richer corpus alongside it — it does not require this cell's outputs beyond
`gold_docs`, `df_sample_clean`, and `distractor_docs` already being in
memory.

In [9]:
corpus = {}
doc_metadata = []

for title, text in gold_docs.items():
    corpus[title] = text
    doc_metadata.append({"title": title, "text": text, "is_gold": True})

for title, text in distractor_docs.items():
    corpus[title] = text
    doc_metadata.append({"title": title, "text": text, "is_gold": False})

print(f"Total corpus size: {len(corpus)}")
print(f"Gold docs: {sum(1 for d in doc_metadata if d['is_gold'])}")
print(f"Distractor docs: {sum(1 for d in doc_metadata if not d['is_gold'])}")

assert sum(1 for d in doc_metadata if d["is_gold"]) == TARGET_N
assert sum(1 for d in doc_metadata if not d["is_gold"]) == N_DISTRACTORS
assert len(corpus) == TARGET_N + N_DISTRACTORS

import pandas as pd
corpus_df = pd.DataFrame(doc_metadata).reset_index(drop=True)
corpus_df["doc_idx"] = corpus_df.index
corpus_df.to_csv("corpus_full.csv", index=False)

overlap = set(gold_docs.keys()) & set(distractor_docs.keys())
print(f"Title overlap between gold/distractor: {len(overlap)}")

corpus_df.head(5)

Total corpus size: 30000
Gold docs: 10000
Distractor docs: 20000
Title overlap between gold/distractor: 0


,title,text,is_gold,doc_idx
0,Nepal–Tibet War (1855–1856),The Nepal–Tibet War (Chinese: 廓藏戰爭; Nepali: ने...,True,0
1,Hara Hara Mahadeva,"Hara Hara Mahadeva (Sanskrit: हर हर महादेव, ro...",True,1
2,Kumaoni people,"Kumaonis, also known as Kumaiye and Kumain (in...",True,2
3,Maggie Gyllenhaal,Margalit Ruth Gyllenhaal Sarsgaard ( JIL-ən-ha...,True,3
4,Hahoe Folk Village,The Hahoe Folk Village (Korean: 안동 하회마을) is a ...,True,4


In [10]:
import shutil, os

os.makedirs("clir_en_dataset", exist_ok=True)

corpus_df.to_csv("clir_en_dataset/corpus_full.csv", index=False)
df_sample_clean.to_csv("clir_en_dataset/query_pairs_final.csv", index=False)

with open("clir_en_dataset/gold_docs_truncated.json", "w", encoding="utf-8") as f:
    json.dump(gold_docs, f, ensure_ascii=False, indent=2)
with open("clir_en_dataset/distractor_docs.json", "w", encoding="utf-8") as f:
    json.dump(distractor_docs, f, ensure_ascii=False, indent=2)

print(f"Queries: {len(df_sample_clean)} | Gold docs: {len(gold_docs)} | Distractors: {len(distractor_docs)} | Total corpus: {len(corpus_df)}")
print(os.listdir("clir_en_dataset"))

Queries: 10000 | Gold docs: 10000 | Distractors: 20000 | Total corpus: 30000
['query_pairs_final.csv', 'gold_docs_truncated.json', 'distractor_docs.json', 'corpus_full.csv']


## Part 2 — Multi-query, difficulty-aware extension

From here on we build a second dataset directory, `clir_en_dataset_v2/`, that
sits on top of the objects already in memory from Part 1:
`df_sample_clean` (qid / ne_title / en_title), `gold_docs`, `distractor_docs`.

### 2.1 Multi-query formulations

For each gold entity we pull three additional sources of Nepali query text,
all mapped to the *same* `qid` / `en_title` gold label:

- **Wikidata aliases** (`ne` language) — alternative names for the entity.
- **Wikipedia redirects** — other Nepali Wikipedia page titles that redirect
  to the same article (informal names, alternate spellings, etc.).
- **Wikidata short description** (`ne` language) — a one-line gloss of the
  entity, giving a query formulated as a description rather than a name.

In [11]:
import requests
import time
import json

WD_API = "https://www.wikidata.org/w/api.php"
NE_API = "https://ne.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "CLIR-research-script/1.0 (your_email@example.com; educational research project)"
}

def safe_get(url, params, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
            if resp.status_code != 200:
                print(f"\nNon-200: {resp.status_code}, retry {attempt+1}")
                time.sleep(2 ** attempt)
                continue
            return resp.json()
        except requests.exceptions.JSONDecodeError:
            print(f"\nJSON decode failed, retry {attempt+1}")
            time.sleep(2 ** attempt)
        except requests.exceptions.RequestException as e:
            print(f"\nRequest exception: {e}, retry {attempt+1}")
            time.sleep(2 ** attempt)
    raise RuntimeError("Max retries exceeded")


def fetch_wikidata_details(qids, batch_size=50, sleep=0.2, save_every=20,
                             save_path="wikidata_entity_details.json"):
    """For each QID, fetch aliases + descriptions (ne/en), the English label,
    and P31 ('instance of') class QIDs — used later for hard-negative mining
    and entity-category metadata."""
    details = {}
    n_batches = (len(qids) + batch_size - 1) // batch_size
    for batch_idx, i in enumerate(range(0, len(qids), batch_size)):
        batch = qids[i:i+batch_size]
        params = {
            "action": "wbgetentities",
            "format": "json",
            "ids": "|".join(batch),
            "props": "aliases|descriptions|claims|labels",
            "languages": "ne|en",
        }
        resp = safe_get(WD_API, params)
        entities = resp.get("entities", {})
        for qid, ent in entities.items():
            aliases_ne = [a["value"] for a in ent.get("aliases", {}).get("ne", [])]
            aliases_en = [a["value"] for a in ent.get("aliases", {}).get("en", [])]
            desc_ne = ent.get("descriptions", {}).get("ne", {}).get("value", "")
            desc_en = ent.get("descriptions", {}).get("en", {}).get("value", "")
            label_en = ent.get("labels", {}).get("en", {}).get("value", "")
            p31_claims = ent.get("claims", {}).get("P31", [])
            instance_of_qids = []
            for c in p31_claims:
                try:
                    instance_of_qids.append(c["mainsnak"]["datavalue"]["value"]["id"])
                except (KeyError, TypeError):
                    continue
            details[qid] = {
                "aliases_ne": aliases_ne,
                "aliases_en": aliases_en,
                "description_ne": desc_ne,
                "description_en": desc_en,
                "label_en": label_en,
                "instance_of_qids": instance_of_qids,
            }
        print(f"Batch {batch_idx+1}/{n_batches}, {len(details)} entities so far", end="\r")
        if (batch_idx + 1) % save_every == 0:
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(details, f, ensure_ascii=False, indent=2)
        time.sleep(sleep)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(details, f, ensure_ascii=False, indent=2)
    print(f"\n\nFetched Wikidata details for {len(details)} entities")
    return details


qid_list = df_sample_clean["qid"].dropna().unique().tolist()
wikidata_details = fetch_wikidata_details(qid_list, batch_size=50, sleep=0.2)

Batch 200/200, 10000 entities so far

Fetched Wikidata details for 10000 entities


In [12]:
def fetch_ne_redirects(ne_titles, batch_size=50, sleep=0.2, save_every=20,
                         save_path="ne_redirects.json"):
    """For each ne_title, find other ne.wikipedia pages that redirect to it —
    these are alternate names/spellings a user might search with."""
    redirects_map = {t: [] for t in ne_titles}
    n_batches = (len(ne_titles) + batch_size - 1) // batch_size
    for batch_idx, i in enumerate(range(0, len(ne_titles), batch_size)):
        batch = ne_titles[i:i+batch_size]
        params = {
            "action": "query",
            "format": "json",
            "titles": "|".join(batch),
            "prop": "redirects",
            "rdlimit": "max",
        }
        resp = safe_get(NE_API, params)
        pages = resp.get("query", {}).get("pages", {})
        for pid, page in pages.items():
            title = page.get("title")
            if title in redirects_map:
                rds = page.get("redirects", [])
                redirects_map[title] = [r["title"] for r in rds]
        print(f"Batch {batch_idx+1}/{n_batches} redirects fetched", end="\r")
        if (batch_idx + 1) % save_every == 0:
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(redirects_map, f, ensure_ascii=False, indent=2)
        time.sleep(sleep)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(redirects_map, f, ensure_ascii=False, indent=2)
    print(f"\n\nFetched redirects for {len(redirects_map)} ne titles")
    return redirects_map


ne_redirects = fetch_ne_redirects(df_sample_clean["ne_title"].tolist(), batch_size=50, sleep=0.2)

Batch 200/200 redirects fetched

Fetched redirects for 10000 ne titles


In [13]:
import pandas as pd

def build_multi_query_df(df_sample_clean, wikidata_details, ne_redirects):
    """One row per (query formulation), all rows for the same entity carry the
    same qid/en_title gold label — this is the multi-query, single-gold-label
    structure the benchmark needs."""
    rows = []
    for _, row in df_sample_clean.iterrows():
        ne_title, en_title, qid = row["ne_title"], row["en_title"], row["qid"]
        wd = wikidata_details.get(qid, {})

        # 1. original title query (same as the Part 1 benchmark)
        rows.append({"qid": qid, "en_title": en_title, "query_text": ne_title, "query_type": "title"})

        # 2. wikipedia redirects (ne) pointing to this article
        for rd in ne_redirects.get(ne_title, []):
            rows.append({"qid": qid, "en_title": en_title, "query_text": rd, "query_type": "redirect"})

        # 3. wikidata aliases (ne)
        for alias in wd.get("aliases_ne", []):
            rows.append({"qid": qid, "en_title": en_title, "query_text": alias, "query_type": "alias"})

        # 4. wikidata short description (ne) as a natural-language query
        desc_ne = wd.get("description_ne", "")
        if desc_ne:
            rows.append({"qid": qid, "en_title": en_title, "query_text": desc_ne, "query_type": "description"})

    multi_query_df = pd.DataFrame(rows)
    multi_query_df = multi_query_df.drop_duplicates(subset=["qid", "query_text", "query_type"]).reset_index(drop=True)
    multi_query_df["query_id"] = multi_query_df.index
    return multi_query_df


multi_query_df = build_multi_query_df(df_sample_clean, wikidata_details, ne_redirects)
print(multi_query_df["query_type"].value_counts())
print(f"\nTotal query formulations: {len(multi_query_df)} across {multi_query_df['qid'].nunique()} gold documents")
multi_query_df.to_csv("multi_query_pool.csv", index=False)
multi_query_df.head(10)

query_type
title          10000
redirect        4277
description     3073
alias           1035
Name: count, dtype: int64

Total query formulations: 18385 across 10000 gold documents


,qid,en_title,query_text,query_type,query_id
0,Q2001528,Nepal–Tibet War (1855–1856),तेश्रो नेपाल तिब्बत युद्ध,title,0
1,Q2001528,Nepal–Tibet War (1855–1856),अन्तीम नेपाल तिब्बत युद्ध,redirect,1
2,Q2001528,Nepal–Tibet War (1855–1856),अन्तिम नेपाल तिब्बत युद्ध,redirect,2
3,Q2001528,Nepal–Tibet War (1855–1856),अन्तिम नेपाल तिब्बत युद्ध,alias,3
4,Q2001528,Nepal–Tibet War (1855–1856),अन्तीम नेपाल तिब्बत युद्ध,alias,4
5,Q119038744,Hara Hara Mahadeva,हर हर महादेव,title,5
6,Q3630074,Kumaoni people,कुमाइँ,title,6
7,Q3630074,Kumaoni people,खस जातिको एक विभाजित इन्डो-आर्य समुह,description,7
8,Q202381,Maggie Gyllenhaal,म्याग्गी जिलेनहाल,title,8
9,Q45865,Hahoe Folk Village,हाहोए लोक गाँउ,title,9


### 2.2 Hard negatives

Rather than sampling distractors uniformly at random (Part 1), we mine
documents that are **topically close** to each gold document:

1. Group gold entities by their primary Wikidata class (first `P31`
   "instance of" value), and resolve human-readable class labels.
2. For each class, run a SPARQL query against the Wikidata Query Service for
   other items of the same class that have an English Wikipedia article —
   these become the hard-negative candidate pool for that class.
3. For classes with too few candidates, fall back to English Wikipedia
   category co-membership (articles that share a category with the gold
   article).
4. Sample a fixed number of hard negatives per gold document from its
   class's candidate pool and fetch their extracts.

In [14]:
from collections import Counter, defaultdict

# Use the single most common P31 class per entity (first listed) as its primary category
qid_to_primary_class = {}
for qid, wd in wikidata_details.items():
    classes = wd.get("instance_of_qids", [])
    if classes:
        qid_to_primary_class[qid] = classes[0]

class_counts = Counter(qid_to_primary_class.values())
print(f"{len(class_counts)} distinct primary classes across gold entities")
print(class_counts.most_common(15))

def fetch_qid_labels(qids, batch_size=50, sleep=0.2):
    """Resolve QIDs (here: Wikidata classes) to their English label."""
    labels = {}
    for i in range(0, len(qids), batch_size):
        batch = qids[i:i+batch_size]
        params = {"action": "wbgetentities", "format": "json", "ids": "|".join(batch),
                   "props": "labels", "languages": "en"}
        resp = safe_get(WD_API, params)
        for qid, ent in resp.get("entities", {}).items():
            labels[qid] = ent.get("labels", {}).get("en", {}).get("value", qid)
        time.sleep(sleep)
    return labels

class_labels = fetch_qid_labels(list(class_counts.keys()))
qid_to_class_label = {qid: class_labels.get(cls, cls) for qid, cls in qid_to_primary_class.items()}
print("\nExample class labels:", list(class_labels.items())[:10])

1494 distinct primary classes across gold entities
[('Q5', 2088), ('Q1530705', 1103), ('Q16521', 673), ('Q486972', 287), ('Q1149652', 210), ('Q7725634', 189), ('Q11424', 153), ('Q13406463', 135), ('Q29467088', 134), ('Q515', 95), ('Q27020041', 77), ('Q4830453', 76), ('Q135408445', 71), ('Q620471', 63), ('Q3624078', 62)]

Example class labels: [('Q198', 'war'), ('Q1671839', 'invocation'), ('Q41710', 'ethnic group'), ('Q5', 'human'), ('Q486972', 'human settlement'), ('Q50846468', 'sports tour'), ('Q16560', 'palace'), ('Q24336031', 'mythical river'), ('Q13406463', 'Wikimedia list article'), ('Q16521', 'taxon')]


In [15]:
SPARQL_URL = "https://query.wikidata.org/sparql"

def fetch_hard_negative_candidates(class_qid, exclude_qids, limit=30, sleep=0.5):
    """Pull other Wikidata items of the same class that have an English
    Wikipedia sitelink, to use as topically-related (hard) negative documents."""
    query = f"""
    SELECT ?item ?enTitle WHERE {{
      ?item wdt:P31 wd:{class_qid} .
      ?sitelink schema:about ?item ;
                schema:isPartOf <https://en.wikipedia.org/> ;
                schema:name ?enTitle .
    }}
    LIMIT {limit}
    """
    resp = requests.get(SPARQL_URL, params={"query": query, "format": "json"},
                         headers=HEADERS, timeout=30)
    time.sleep(sleep)
    out = []
    if resp.status_code == 200:
        for b in resp.json().get("results", {}).get("bindings", []):
            qid = b["item"]["value"].rsplit("/", 1)[-1]
            if qid not in exclude_qids:
                out.append({"qid": qid, "en_title": b["enTitle"]["value"]})
    else:
        print(f"SPARQL non-200 for class {class_qid}: {resp.status_code}")
    return out

gold_qid_set = set(df_sample_clean["qid"])
hard_negative_candidates = defaultdict(list)  # class_qid -> [{qid, en_title}, ...]

for class_qid in class_counts:
    try:
        cands = fetch_hard_negative_candidates(class_qid, gold_qid_set, limit=30)
        hard_negative_candidates[class_qid] = cands
        print(f"Class {class_qid} ({class_labels.get(class_qid, '?')}): {len(cands)} candidates", end="\r")
    except requests.exceptions.RequestException as e:
        print(f"\nSPARQL request failed for {class_qid}: {e}")
    time.sleep(0.5)

with open("hard_negative_candidates.json", "w", encoding="utf-8") as f:
    json.dump(hard_negative_candidates, f, ensure_ascii=False, indent=2)
print(f"\n\nCandidate pools built for {len(hard_negative_candidates)} classes")

Class Q129133491 (ethnological term): 8 candidateslassification): 1 candidates): 16 candidates candidates
SPARQL request failed for Q13433827: HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=30)
Class Q81989119 (video game distribution platform): 30 candidatesandidatessesestes

Candidate pools built for 1493 classes


In [16]:
EN_API = "https://en.wikipedia.org/w/api.php"

def fetch_category_members(en_title, limit=20):
    """Fallback hard-negative source for classes with too few SPARQL
    candidates: other articles sharing an English Wikipedia category with the
    gold article."""
    cat_params = {"action": "query", "format": "json", "titles": en_title,
                   "prop": "categories", "cllimit": "max"}
    resp = safe_get(EN_API, cat_params)
    pages = resp.get("query", {}).get("pages", {})
    cats = []
    for page in pages.values():
        cats.extend(c["title"] for c in page.get("categories", []) if "stub" not in c["title"].lower())
    if not cats:
        return []
    cat = cats[0]
    member_params = {"action": "query", "format": "json", "list": "categorymembers",
                       "cmtitle": cat, "cmlimit": limit, "cmnamespace": 0}
    resp2 = safe_get(EN_API, member_params)
    return [m["title"] for m in resp2.get("query", {}).get("categorymembers", [])]

MIN_CANDIDATES_PER_CLASS = 5
category_fallback_used = []
for qid in df_sample_clean["qid"]:
    cls = qid_to_primary_class.get(qid)
    if cls is None or len(hard_negative_candidates.get(cls, [])) < MIN_CANDIDATES_PER_CLASS:
        en_title = df_sample_clean.loc[df_sample_clean["qid"] == qid, "en_title"].iloc[0]
        members = fetch_category_members(en_title, limit=20)
        category_fallback_used.append({"qid": qid, "en_title": en_title, "category_members": members})
        time.sleep(0.2)

print(f"Used category fallback for {len(category_fallback_used)} gold entities")
with open("category_fallback.json", "w", encoding="utf-8") as f:
    json.dump(category_fallback_used, f, ensure_ascii=False, indent=2)


Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3

Non-200: 429, retry 4

Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3

Non-200: 429, retry 4

Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3

Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3

Non-200: 429, retry 4

Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3

Non-200: 429, retry 4

Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3

Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3

Non-200: 429, retry 4

Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3

Non-200: 429, retry 4
Used category fallback for 826 gold entities


In [17]:
import random

N_HARD_NEG_PER_GOLD = 3

def sample_hard_negatives_for_gold(qid, en_title, n=N_HARD_NEG_PER_GOLD):
    cls = qid_to_primary_class.get(qid)
    pool = [c["en_title"] for c in hard_negative_candidates.get(cls, [])] if cls else []
    if len(pool) < n:
        fb = next((f["category_members"] for f in category_fallback_used if f["qid"] == qid), [])
        pool = list(dict.fromkeys(pool + fb))  # preserve order, dedup
    pool = [t for t in pool if t != en_title]
    random.seed(hash(qid) % (2**32))
    return random.sample(pool, min(n, len(pool)))

hard_negative_titles = {}  # gold en_title -> [hard negative en_titles]
for _, row in df_sample_clean.iterrows():
    hard_negative_titles[row["en_title"]] = sample_hard_negatives_for_gold(row["qid"], row["en_title"])

all_hard_neg_titles = sorted(set(t for lst in hard_negative_titles.values() for t in lst))
print(f"{len(all_hard_neg_titles)} unique hard-negative articles to fetch extracts for")

hard_negative_docs = {}
for i in range(0, len(all_hard_neg_titles), 50):
    batch = all_hard_neg_titles[i:i+50]
    extracts = fetch_extracts(batch, batch_size=50)
    for t, text in extracts.items():
        if text and len(text.split()) >= 20:
            hard_negative_docs[t] = truncate_tokens(text, 200)
    print(f"Fetched {len(hard_negative_docs)}/{len(all_hard_neg_titles)} hard-negative extracts", end="\r")
    time.sleep(0.2)

with open("hard_negative_docs.json", "w", encoding="utf-8") as f:
    json.dump(hard_negative_docs, f, ensure_ascii=False, indent=2)
print(f"\n\nCollected {len(hard_negative_docs)} hard-negative documents")

9072 unique hard-negative articles to fetch extracts for
Batch 1: 20/50 fetched
Batch 1: 20/50 fetchedegative extracts
Batch 1: 20/50 fetchedegative extracts
Batch 1: 20/50 fetchedegative extracts
Batch 1: 20/50 fetchedegative extracts
Batch 1: 20/50 fetchedegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch 1: 20/50 fetchednegative extracts
Batch

### 2.3 Per-instance metadata

For every query formulation we attach:

- **`alias_count`** — how many Wikidata aliases (ne) exist for the entity.
- **`ambiguity_level`** — how many distinct Wikidata items share the query's
  exact label (via `wbsearchentities`); bucketed into low / medium / high.
- **`entity_category`** — the entity's primary Wikidata class label (from
  2.2), e.g. "human", "film", "human settlement".
- **`article_length`** — token count of the gold document.
- **`lexical_overlap`** — normalized string-similarity between the query
  text and the gold document's English title. Because queries are in
  Devanagari script and titles are in Latin script, this is expected to sit
  near zero for almost all instances — which is itself a useful diagnostic:
  it confirms that matching cannot rely on surface-form overlap, and any
  instance where it is *not* near zero (loanwords, numbers, acronyms
  embedded in the Nepali text) flags a potentially easier cross-lingual
  match worth analyzing separately.

In [18]:
import difflib

def fetch_label_ambiguity(query_text, language="ne", sleep=0.2):
    """Count how many distinct Wikidata items carry this exact label —
    a simple proxy for how ambiguous a query formulation is."""
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "search": query_text,
        "language": language,
        "type": "item",
        "limit": 10,
    }
    resp = safe_get(WD_API, params)
    time.sleep(sleep)
    exact_matches = [r for r in resp.get("search", []) if r.get("label") == query_text]
    return len(exact_matches)

def bucket_ambiguity(n_matches):
    if n_matches <= 1:
        return "low"
    elif n_matches <= 3:
        return "medium"
    else:
        return "high"

def lexical_overlap_score(query_text, en_title):
    """Character-level similarity between the query text and the gold
    document's English title (SequenceMatcher ratio, 0-1)."""
    return difflib.SequenceMatcher(None, query_text.lower(), en_title.lower()).ratio()

gold_doc_lengths = {title: len(text.split()) for title, text in gold_docs.items()}
alias_counts = {qid: len(wd.get("aliases_ne", [])) for qid, wd in wikidata_details.items()}

In [19]:
# Ambiguity lookups are one wbsearchentities call per unique query text —
# cache aggressively since many rows share a title/alias.
ambiguity_cache = {}

def get_ambiguity_bucket(query_text):
    if query_text not in ambiguity_cache:
        n_matches = fetch_label_ambiguity(query_text, language="ne", sleep=0.1)
        ambiguity_cache[query_text] = bucket_ambiguity(n_matches)
    return ambiguity_cache[query_text]

metadata_rows = []
for _, row in multi_query_df.iterrows():
    qid, en_title, query_text = row["qid"], row["en_title"], row["query_text"]
    metadata_rows.append({
        "query_id": row["query_id"],
        "alias_count": alias_counts.get(qid, 0),
        "ambiguity_level": get_ambiguity_bucket(query_text),
        "entity_category": qid_to_class_label.get(qid, "unknown"),
        "article_length": gold_doc_lengths.get(en_title, 0),
        "lexical_overlap": round(lexical_overlap_score(query_text, en_title), 4),
    })
    if len(metadata_rows) % 500 == 0:
        print(f"Enriched {len(metadata_rows)}/{len(multi_query_df)} query rows", end="\r")

metadata_df = pd.DataFrame(metadata_rows)
benchmark_df = multi_query_df.merge(metadata_df, on="query_id")
print(f"\n\nEnriched benchmark: {len(benchmark_df)} rows")
benchmark_df.to_csv("multi_query_benchmark_enriched.csv", index=False)
benchmark_df.head(10)


Non-200: 429, retry 1

Non-200: 429, retry 1

Non-200: 429, retry 2
Enriched 500/18385 query rows
Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 1

Non-200: 429, retry 2
Enriched 1000/18385 query rows
Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3
Enriched 1500/18385 query rows
Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 1
Enriched 2000/18385 query rows
Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3
Enriched 2500/18385 query rows
Non-200: 429, retry 1

Non-200: 429, retry 2
Enriched 3000/18385 query rows
Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 3
Enriched 3500/18385 query rows
Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 1
Enriched 4000/18385 query rows
Non-200: 429, retry 1

Non-200: 429, retry 2

Non-200: 429, retry 1

Non-200: 429, retry 2
Enriched 4500/18385 query rows
Non-200: 429, retry 1

Non-200: 429, ret

,qid,en_title,query_text,query_type,query_id,alias_count,ambiguity_level,entity_category,article_length,lexical_overlap
0,Q2001528,Nepal–Tibet War (1855–1856),तेश्रो नेपाल तिब्बत युद्ध,title,0,2,low,war,53,0.0769
1,Q2001528,Nepal–Tibet War (1855–1856),अन्तीम नेपाल तिब्बत युद्ध,redirect,1,2,low,war,53,0.0769
2,Q2001528,Nepal–Tibet War (1855–1856),अन्तिम नेपाल तिब्बत युद्ध,redirect,2,2,low,war,53,0.0769
3,Q2001528,Nepal–Tibet War (1855–1856),अन्तिम नेपाल तिब्बत युद्ध,alias,3,2,low,war,53,0.0769
4,Q2001528,Nepal–Tibet War (1855–1856),अन्तीम नेपाल तिब्बत युद्ध,alias,4,2,low,war,53,0.0769
5,Q119038744,Hara Hara Mahadeva,हर हर महादेव,title,5,0,low,invocation,96,0.1333
6,Q3630074,Kumaoni people,कुमाइँ,title,6,0,low,ethnic group,57,0.0000
7,Q3630074,Kumaoni people,खस जातिको एक विभाजित इन्डो-आर्य समुह,description,7,0,low,ethnic group,57,0.0400
8,Q202381,Maggie Gyllenhaal,म्याग्गी जिलेनहाल,title,8,0,low,human,200,0.0588
9,Q45865,Hahoe Folk Village,हाहोए लोक गाँउ,title,9,0,low,human settlement,93,0.1250


### 2.4 Assemble and save the v2 corpus (gold + hard negatives + random distractors)

In [21]:
import os

corpus_v2 = {}
doc_metadata_v2 = []

for title, text in gold_docs.items():
    corpus_v2[title] = text
    doc_metadata_v2.append({"title": title, "text": text, "doc_type": "gold"})

for title, text in hard_negative_docs.items():
    if title not in corpus_v2:
        corpus_v2[title] = text
        doc_metadata_v2.append({"title": title, "text": text, "doc_type": "hard_negative"})

for title, text in distractor_docs.items():
    if title not in corpus_v2:
        corpus_v2[title] = text
        doc_metadata_v2.append({"title": title, "text": text, "doc_type": "random_distractor"})

corpus_v2_df = pd.DataFrame(doc_metadata_v2).reset_index(drop=True)
corpus_v2_df["doc_idx"] = corpus_v2_df.index

print(f"Total v2 corpus size: {len(corpus_v2_df)}")
print(corpus_v2_df["doc_type"].value_counts())

os.makedirs("clir_en_dataset_v2", exist_ok=True)
corpus_v2_df.to_csv("clir_en_dataset_v2/corpus_full_v2.csv", index=False)
benchmark_df.to_csv("clir_en_dataset_v2/multi_query_benchmark_enriched.csv", index=False)

with open("clir_en_dataset_v2/hard_negative_titles_by_gold.json", "w", encoding="utf-8") as f:
    json.dump(hard_negative_titles, f, ensure_ascii=False, indent=2)
with open("clir_en_dataset_v2/wikidata_entity_details.json", "w", encoding="utf-8") as f:
    json.dump(wikidata_details, f, ensure_ascii=False, indent=2)

print("\nSaved files:", os.listdir("clir_en_dataset_v2"))

Total v2 corpus size: 33464
doc_type
random_distractor    19982
gold                 10000
hard_negative         3482
Name: count, dtype: int64

Saved files: ['hard_negative_titles_by_gold.json', 'wikidata_entity_details.json', 'multi_query_benchmark_enriched.csv', 'corpus_full_v2.csv']


### 2.5 Dataset summary

A quick sanity check over the enriched benchmark: query-type distribution,
ambiguity distribution, and mean lexical overlap by query type (should be
near zero throughout, confirming genuine cross-lingual — not surface-form —
matching is required, with `alias`/`description` types typically showing
slightly different overlap patterns than `title`).

In [22]:
print("Query formulations by type:")
print(benchmark_df["query_type"].value_counts())

print("\nAmbiguity level distribution:")
print(benchmark_df["ambiguity_level"].value_counts())

print("\nTop 10 entity categories:")
print(benchmark_df["entity_category"].value_counts().head(10))

print("\nMean lexical overlap by query type:")
print(benchmark_df.groupby("query_type")["lexical_overlap"].mean().sort_values(ascending=False))

print(f"\nGold documents: {df_sample_clean['qid'].nunique()}")
print(f"Total query formulations: {len(benchmark_df)}")
print(f"Hard negatives: {len(hard_negative_docs)}")
print(f"Random distractors: {len(distractor_docs)}")
print(f"Total corpus size: {len(corpus_v2_df)}")

Query formulations by type:
query_type
title          10000
redirect        4277
description     3073
alias           1035
Name: count, dtype: int64

Ambiguity level distribution:
ambiguity_level
low       18336
medium       43
high          6
Name: count, dtype: int64

Top 10 entity categories:
entity_category
human                          4043
ward of Nepal                  1775
taxon                           926
unknown                         923
district of India               503
human settlement                495
literary work                   349
rural municipality of Nepal     289
film                            270
Wikimedia list article          237
Name: count, dtype: int64

Mean lexical overlap by query type:
query_type
alias          0.084522
title          0.057918
redirect       0.054993
description    0.043243
Name: lexical_overlap, dtype: float64

Gold documents: 10000
Total query formulations: 18385
Hard negatives: 3496
Random distractors: 20000
Total corpus size